# Generative AI · Assignment 1 (starter)
### Latent Spaces, and Why GANs Are Hard to Train

**FAST-NUCES Lahore · Fall 2026 **

This notebook gives you the plumbing so you can spend your two weeks on the parts that
carry the marks. Everything marked **`# TODO`** is yours to write. Do not delete the
logging hook in the training loop — several questions ask for a number that only exists
in your CSV.

Read the handout first. Your seed, your held-out digit, your six digits and your second
CelebA attribute all come from your roll number (§1.2).

---
## 0 · Setup

In [1]:
# ---- YOUR ROLL NUMBER -------------------------------------------------------
ROLL = "21L-3333"          # TODO: put your real roll number here
# -----------------------------------------------------------------------------

R    = int(ROLL[-2:])                 # last two digits
SEED = int(ROLL[-4:])                 # last four digits  -> the seed for every run

HELD_OUT_DIGIT = R % 10                                  # Part B anomaly class
MY_DIGITS      = sorted({(R + 3 * k) % 10 for k in range(6)})
for d in range(10):                                      # top up if the set is short
    if len(MY_DIGITS) == 6: break
    if d not in MY_DIGITS: MY_DIGITS = sorted(MY_DIGITS + [d])
ATTRS      = ["Eyeglasses", "Male", "Young", "Blond_Hair", "Wearing_Hat", "Mouth_Slightly_Open"]
MY_ATTR    = ATTRS[R % 6]

print(f"roll {ROLL}   R={R}   SEED={SEED}")
print(f"held-out digit : {HELD_OUT_DIGIT}")
print(f"my six digits  : {MY_DIGITS}")
print(f"my attribute   : Smiling  +  {MY_ATTR}")

roll 21L-3333   R=33   SEED=3333
held-out digit : 3
my six digits  : [2, 3, 5, 6, 8, 9]
my attribute   : Smiling  +  Blond_Hair


In [2]:
import os, json, math, random, csv, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from scipy import linalg
from sklearn.metrics import roc_auc_score, precision_recall_curve

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(DEV, torch.__version__)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
for d in ["figures", "logs", "ckpt"]:
    os.makedirs(d, exist_ok=True)

METRICS = {}                       # everything you quote in the report goes in here
def record(key, value):
    METRICS[key] = value
    json.dump(METRICS, open("metrics.json", "w"), indent=2, default=float)
    print(f"  {key} = {value}")

cpu 2.11.0+cpu


### 0.1 · Plot helpers

In [3]:
def show_grid(imgs, nrow=8, title=None, save=None, figsize=(10, 4)):
    """imgs: (N,C,H,W) tensor in [0,1] (or [-1,1] -- it is detected and rescaled)."""
    x = imgs.detach().cpu().float()
    if x.min() < -0.01: x = (x + 1) / 2
    g = torchvision.utils.make_grid(x.clamp(0, 1), nrow=nrow, padding=2, pad_value=1)
    plt.figure(figsize=figsize)
    plt.imshow(g.permute(1, 2, 0).squeeze(), cmap="gray" if g.shape[0] == 1 else None)
    plt.axis("off")
    if title: plt.title(title, fontsize=11)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=140, bbox_inches="tight")
    plt.show()


def strip(imgs, labels=None, title=None, save=None):
    """One row of images with optional per-image captions -- use this for interpolations."""
    x = imgs.detach().cpu().float()
    if x.min() < -0.01: x = (x + 1) / 2
    n = x.shape[0]
    fig, ax = plt.subplots(1, n, figsize=(1.35 * n, 1.75))
    for i in range(n):
        im = x[i].clamp(0, 1).permute(1, 2, 0).squeeze()
        ax[i].imshow(im, cmap="gray" if x.shape[1] == 1 else None)
        ax[i].axis("off")
        if labels is not None: ax[i].set_title(str(labels[i]), fontsize=8)
    if title: fig.suptitle(title, fontsize=11, y=1.06)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=140, bbox_inches="tight")
    plt.show()

### 0.2 · CSV logger for training runs — **do not delete**

In [4]:
class RunLog:
    """Writes one row per iteration. Several questions in the handout ask for a specific
    iteration number that can only come from this file."""
    COLS = ["iter", "d_loss", "g_loss", "D_x", "D_G_z", "g_grad_norm"]

    def __init__(self, name):
        self.path = f"logs/{name}.csv"
        self.f = open(self.path, "w", newline="")
        self.w = csv.writer(self.f); self.w.writerow(self.COLS)
        self.rows = []

    def __call__(self, it, d_loss, g_loss, D_x, D_G_z, g_grad_norm):
        row = [it, float(d_loss), float(g_loss), float(D_x), float(D_G_z), float(g_grad_norm)]
        self.rows.append(row); self.w.writerow(row)
        if it % 200 == 0: self.f.flush()

    def close(self):
        self.f.flush(); self.f.close()
        return np.array(self.rows)


def grad_norm(module):
    """L2 norm of the gradient of the FIRST parameter tensor of `module`.
    Call it after loss.backward() and before optimiser.step()."""
    for prm in module.parameters():
        if prm.grad is not None:
            return prm.grad.detach().norm(2).item()
    return 0.0


def collapse_iteration(log_array, frac=0.01, ref_iter=100):
    """Part C1: the first iteration at which the generator gradient norm falls below
    `frac` of its value at iteration `ref_iter`.  Returns (iteration, ref_value)."""
    it, gn = log_array[:, 0], log_array[:, 5]
    ref = float(np.interp(ref_iter, it, gn))
    below = np.where((it > ref_iter) & (gn < frac * ref))[0]
    return (int(it[below[0]]) if len(below) else None), ref

---
## 1 · Data

MNIST downloads automatically. CelebA is the one that needs care — Colab's `torchvision`
CelebA download is frequently rate-limited, so the cell below tries `kagglehub` first and
falls back to torchvision. If both fail, download `img_align_celeba.zip` and
`list_attr_celeba.txt` by hand into `data/celeba/`.

In [5]:
# ---------------- MNIST ----------------
mnist_tf = transforms.ToTensor()                        # [0,1], shape (1,28,28)
mnist_tr = torchvision.datasets.MNIST("data", train=True,  download=True, transform=mnist_tf)
mnist_te = torchvision.datasets.MNIST("data", train=False, download=True, transform=mnist_tf)

def subset_by_digits(ds, digits):
    idx = [i for i, y in enumerate(ds.targets.tolist()) if y in digits]
    return Subset(ds, idx)

def exclude_digit(ds, digit):
    idx = [i for i, y in enumerate(ds.targets.tolist()) if y != digit]
    return Subset(ds, idx)

mnist_mine = subset_by_digits(mnist_tr, MY_DIGITS)       # Parts C, D, E
print(f"MNIST train {len(mnist_tr)} | my six digits {len(mnist_mine)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 339kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.15MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.04MB/s]

MNIST train 60000 | my six digits 35228


In [6]:
# ---------------- CelebA 64x64 ----------------
CELEBA_ROOT = "data/celeba"
IMG, N_CELEBA = 64, 30000          # 30k images is plenty for this assignment

celeba_tf = transforms.Compose([
    transforms.CenterCrop(148),     # the standard CelebA crop: drops background, keeps the face
    transforms.Resize(IMG),
    transforms.ToTensor(),
])

def load_celeba():
    """Returns (images_tensor (N,3,64,64) in [0,1], attr_dataframe-like dict)."""
    try:
        import kagglehub
        p = kagglehub.dataset_download("jessicali9530/celeba-dataset")
        root = os.path.join(p, "img_align_celeba", "img_align_celeba")
        attr = os.path.join(p, "list_attr_celeba.csv")
        return root, attr
    except Exception as e:
        print("kagglehub failed:", e)
        print("Falling back to torchvision (may be rate-limited).")
        ds = torchvision.datasets.CelebA("data", split="train", download=True,
                                         transform=celeba_tf, target_type="attr")
        return ds, None

# TODO: run load_celeba(), build a tensor of N_CELEBA images and a dict
#       {attribute_name: bool array}. You need at least "Smiling" and MY_ATTR.
#       Cache the tensor to disk -- you will reload it several times.
#
#   celeba_x    : FloatTensor (N,3,64,64) in [0,1]
#   celeba_attr : dict[str] -> np.array of bool, length N

---
## 2 · Part A — AE vs VAE on CelebA

One architecture, used twice. Do not let the two models differ in anything except the
reparameterisation, the two heads and the KL term.

In [ ]:
LATENT = 64

class ConvEncoder(nn.Module):
    """64 -> 32 -> 16 -> 8 -> 4, then flatten. Outputs `out_dim` numbers."""
    def __init__(self, out_dim, ch=(32, 64, 128, 256)):
        super().__init__()
        layers, c_in = [], 3
        for c in ch:
            layers += [nn.Conv2d(c_in, c, 4, 2, 1), nn.BatchNorm2d(c), nn.LeakyReLU(0.2, True)]
            c_in = c
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(ch[-1] * 4 * 4, out_dim)

    def forward(self, x):
        return self.head(self.body(x).flatten(1))


class ConvDecoder(nn.Module):
    def __init__(self, in_dim, ch=(256, 128, 64, 32)):
        super().__init__()
        self.fc = nn.Linear(in_dim, ch[0] * 4 * 4)
        self.c0 = ch[0]
        layers = []
        for a, b_ in zip(ch, ch[1:]):
            layers += [nn.ConvTranspose2d(a, b_, 4, 2, 1), nn.BatchNorm2d(b_), nn.ReLU(True)]
        layers += [nn.ConvTranspose2d(ch[-1], 3, 4, 2, 1), nn.Sigmoid()]
        self.body = nn.Sequential(*layers)

    def forward(self, z):
        return self.body(self.fc(z).view(-1, self.c0, 4, 4))


class AE(nn.Module):
    def __init__(self, d=LATENT):
        super().__init__()
        self.enc, self.dec = ConvEncoder(d), ConvDecoder(d)
    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z


class VAE(nn.Module):
    def __init__(self, d=LATENT):
        super().__init__()
        self.enc, self.dec, self.d = ConvEncoder(2 * d), ConvDecoder(d), d

    def encode(self, x):
        h = self.enc(x)
        return h[:, :self.d], h[:, self.d:]           # mu, logvar

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * (0.5 * logvar).exp()   # reparameterisation
        return self.dec(z), mu, logvar


def kl_term(mu, logvar):
    """D_KL( q(z|x) || N(0,I) ) summed over dimensions, averaged over the batch.
    This is the closed form from the lecture: 1/2 * sum( sigma^2 + mu^2 - 1 - log sigma^2 )."""
    # TODO: one line. Watch the sign, and remember logvar is log(sigma^2), not log(sigma).
    raise NotImplementedError

In [ ]:
# TODO: A1 -- train the AE and the VAE.
#   Same optimiser, same epochs, same batch size. Log train/val reconstruction MSE.
#   Save both to ckpt/.  Figure: figures/A1_recon.png  (originals above reconstructions)
#
#   record("A1_ae_val_mse", ...)   record("A1_vae_val_mse", ...)

In [ ]:
# ---- A2: what the VAE can do ----
@torch.no_grad()
def interpolate(decode, z1, z2, n=11):
    """Decode n evenly spaced points on the straight line from z1 to z2."""
    t = torch.linspace(0, 1, n, device=z1.device).view(-1, 1)
    return decode((1 - t) * z1.view(1, -1) + t * z2.view(1, -1)), t.flatten().tolist()

# TODO: A2 -- pick one Smiling=+1 and one Smiling=-1 image, encode both with the VAE and
#   interpolate the MEANS:
#       mu1, mu2 = vae.encode(x1)[0], vae.encode(x2)[0]
#       row, ts  = interpolate(vae.dec, mu1, mu2)
#   figures/A2_vae_interp.png    then repeat for MY_ATTR.
#   In the report: is every frame a plausible face, and does the attribute change
#   gradually along the strip or jump at one point?

In [ ]:
# ---- A3: what the AE cannot do ----
# TODO: draw 16 codes from N(0, I) -- the SAME 16 for both models -- and decode them with
#   ae.dec and with vae.dec. Show the two grids side by side.
#   figures/A3_prior_samples.png
#
#   Then decode 16 codes taken from the AE's OWN encoder, so the reader can see the
#   decoder is not broken: it works fine on the codes it was trained on. The prior
#   samples are simply in a part of the space no encoder ever visited.

---
## 3 · Part B — denoising (MNIST)

In [ ]:
def psnr(a, b_, data_range=1.0):
    """a, b_: tensors in [0,1]. Returns mean PSNR in dB over the batch."""
    mse = ((a - b_) ** 2).flatten(1).mean(1).clamp_min(1e-12)
    return (10 * torch.log10(data_range ** 2 / mse)).mean().item()

class SmallAE(nn.Module):
    """28x28 -> LATENT_M -> 28x28. Small enough to train in a couple of minutes."""
    def __init__(self, d=32):
        super().__init__()
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(True), nn.Linear(256, d))
        self.dec = nn.Sequential(nn.Linear(d, 256), nn.ReLU(True), nn.Linear(256, 784),
                                 nn.Sigmoid(), nn.Unflatten(1, (1, 28, 28)))
    def forward(self, x):
        return self.dec(self.enc(x))

# TODO: B1 -- denoising AE. Noisy input x + 0.3*eps clipped to [0,1], clean target.
#       record("B1_psnr_noisy", ...)  record("B1_psnr_denoised", ...)
#       figures/B1_denoise.png : three rows -- clean / noisy / denoised
#


---
## 4 · Parts C–D — GANs on your six digits

`GAN_SHAPES` below is a working DCGAN for 28×28. Part C1 deliberately breaks it; Part D
puts it back together.

In [ ]:
ZDIM = 64

class G28(nn.Module):
    """z -> 7x7 -> 14x14 -> 28x28. Set batchnorm=False for the Part C3 collapse recipe."""
    def __init__(self, zdim=ZDIM, ch=128, batchnorm=True):
        super().__init__()
        def bn(c): return nn.BatchNorm2d(c) if batchnorm else nn.Identity()
        self.fc = nn.Sequential(nn.Linear(zdim, ch * 7 * 7), nn.Unflatten(1, (ch, 7, 7)),
                                bn(ch), nn.ReLU(True))
        self.body = nn.Sequential(
            nn.ConvTranspose2d(ch, ch // 2, 4, 2, 1), bn(ch // 2), nn.ReLU(True),   # 14
            nn.ConvTranspose2d(ch // 2, 1, 4, 2, 1), nn.Tanh(),                     # 28
        )
    def forward(self, z): return self.body(self.fc(z))


class D28(nn.Module):
    def __init__(self, ch=64, spectral=False):
        super().__init__()
        w = (lambda m: nn.utils.spectral_norm(m)) if spectral else (lambda m: m)
        self.body = nn.Sequential(
            w(nn.Conv2d(1, ch, 4, 2, 1)), nn.LeakyReLU(0.2, True),                 # 14
            w(nn.Conv2d(ch, ch * 2, 4, 2, 1)), nn.LeakyReLU(0.2, True),            # 7
            nn.Flatten(), w(nn.Linear(ch * 2 * 7 * 7, 1)),
        )
    def forward(self, x): return self.body(x)      # LOGITS -- apply sigmoid yourself


def to_pm1(x): return x * 2 - 1        # MNIST arrives in [0,1]; G outputs tanh
def to_01(x):  return (x + 1) / 2

In [ ]:
# ============================================================================
#  TODO: C1 -- the SATURATING generator loss, with D deliberately too strong.
#
#  loader: DataLoader(mnist_mine, batch_size=128, shuffle=True, drop_last=True)
#  D steps per G step : 5
#  lr : D = 8e-4, G = 2e-4       (four times, as the handout says)
#  L_D = -[ log D(x) + log(1 - D(G(z))) ]        <- as usual
#  L_G = log(1 - D(G(z)))                        <- the SATURATING one
#
#  Skeleton of the inner loop -- keep the logging call:
#
#     ...
#     g_loss.backward()
#     gn = grad_norm(G.fc)                       # <-- FIRST layer of the generator
#     opt_G.step()
#     log(it, d_loss.item(), g_loss.item(), D_x, D_G_z, gn)
#
#  Afterwards:
#     arr = log.close()
#     it_dead, ref = collapse_iteration(arr)
#     record("C1_ref_gradnorm_at_100", ref)
#     record("C1_iteration_gradient_died", it_dead)
#
#  Figures: D(x) and D(G(z)) vs iteration; grad norm on a LOG y-axis;
#           64-sample grids at it_dead and at the end.
# ============================================================================

In [ ]:
# ============================================================================
#  TODO: C2 -- identical run, one line changed:  L_G = -log D(G(z))
#  set_seed() again first so the two runs are comparable.
#  figures/C2_overlay.png : both grad-norm curves on one log axis.
# ============================================================================

### 4.1 · The classifier — you need it for mode counting, IS and FID

In [ ]:
class MnistCNN(nn.Module):
    """Trains to ~99% on a six-digit subset in about a minute.
    `features()` gives the 128-d penultimate vector -- that is your FID feature space."""
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2),   # 14
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2),  # 7
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(True))
        self.head = nn.Linear(128, n_classes)
    def features(self, x): return self.conv(x)
    def forward(self, x):  return self.head(self.conv(x))

# TODO: train MnistCNN on subset_by_digits(mnist_tr, MY_DIGITS) for 2-3 epochs.
#       Labels must be remapped to 0..5:  {d: i for i, d in enumerate(MY_DIGITS)}
#       Report test accuracy; save to ckpt/clf.pt. Anything below 97% is not good enough
#       to trust the metrics that depend on it.

In [ ]:
@torch.no_grad()
def classify(clf, imgs, bs=500):
    """imgs in [0,1], (N,1,28,28). Returns (probs (N,K), features (N,128))."""
    P, Fe = [], []
    for i in range(0, len(imgs), bs):
        xb = imgs[i:i+bs].to(DEV)
        f = clf.features(xb); p = clf.head(f).softmax(1)
        P.append(p.cpu()); Fe.append(f.cpu())
    return torch.cat(P).numpy(), torch.cat(Fe).numpy()


def mode_stats(probs, k=6, tol=0.01):
    """Part C3 / D. probs: (N,K) softmax over your six digits.
    Returns (histogram, modes_covered, reverse_KL to the uniform target)."""
    pred = probs.argmax(1)
    hist = np.bincount(pred, minlength=k).astype(float)
    p_gen = hist / hist.sum()
    modes = int((p_gen >= tol).sum())
    q = np.full(k, 1 / k)                                  # uniform target
    m = p_gen > 0
    rev_kl = float((p_gen[m] * np.log(p_gen[m] / q[m])).sum())   # D_KL(p_gen || p_data)
    return p_gen, modes, rev_kl

In [ ]:
# ============================================================================
#  TODO: C3 -- force a collapse, then MEASURE it.
#    recipe: ZDIM=2, G28(batchnorm=False), FIVE G steps per D step, lr_G=2e-3, lr_D=2e-4.
#    (The generator races ahead of a stale D and piles every z onto the one sample that
#     fools it. A partial collapse counts, as long as you measure it.)
#    p_gen, modes, rev_kl = mode_stats(classify(clf, samples_01)[0])
#    record("C3_modes_covered", modes) ; record("C3_reverse_kl", rev_kl)
#    figures/C3_hist.png  and  figures/C3_samples.png (64 samples)
#
#  TODO: D1 -- the proper DCGAN. Adam(2e-4, betas=(0.5, 0.999)), non-saturating loss,
#    1:1 steps, batchnorm on. Fixed-noise grid at 4 points in training, SAME z each time.
#    Then the same mode_stats measurement.
#
#  TODO: D2 -- one stabiliser: WGAN-GP, D28(spectral=True), or minibatch discrimination.
#    Implement it yourself. Re-measure. Fill in the four-column table from the handout.
#    (Minibatch discrimination goes in D, not G.)
# ============================================================================

---
## 5 · Part E — Inception Score and FID

Both from your own classifier: softmax for IS, the 128-d penultimate features for FID.
Write the formulas yourself. You may check against `pytorch-fid` at the very end, and if
you do, say so in the report.

In [ ]:
def inception_score(probs, splits=10):
    """IS = exp( E_x [ D_KL( p(y|x) || p(y) ) ] ), computed over `splits` disjoint chunks.
    Returns (mean, std)."""
    # TODO
    #   for each split:
    #       p_yx = that chunk                       (n, K)
    #       p_y  = p_yx.mean(0, keepdims=True)      (1, K)     the marginal
    #       kl   = (p_yx * (log p_yx - log p_y)).sum(1).mean()
    #       score = exp(kl)
    raise NotImplementedError


def frechet_distance(feat_real, feat_gen, eps=1e-6):
    """FID = ||mu_r - mu_g||^2 + Tr( S_r + S_g - 2 (S_r S_g)^(1/2) )."""
    # TODO
    #   mu  = feat.mean(0) ;  S = np.cov(feat, rowvar=False)
    #   covmean = linalg.sqrtm(S_r @ S_g)
    #   sqrtm of a product of two PSD matrices can come back with a tiny imaginary part
    #   for numerical reasons -- take .real, and be ready to explain WHY in the report.
    #   If it is not almost-real (np.abs(covmean.imag).max() large), something is wrong.
    raise NotImplementedError

In [ ]:
# ============================================================================
#  TODO: E1 / E2 -- report IS and FID for five sets:
#        real | C1 | C3 (collapsed) | D1 | D2
#        Reference statistics for FID come from the real TRAINING set.
#
#  TODO: E3 (a) the floor -- FID between two disjoint halves of the real TEST set.
#        record("E3_fid_floor", ...). It will not be zero. Say what it measures.
#
#  TODO: E3 (b) monotonicity -- FID(real, real + sigma*noise) for
#        sigma in [0, 0.1, 0.25, 0.5, 1.0]. Plot. A non-monotone curve means a bug.
#
#  TODO: E3 (c) the question. Your C3 model probably scores a decent IS and a bad FID.
#        Five sentences, using YOUR two numbers, on what each metric cannot see.
# ============================================================================

---
## 6 · Before you submit

- [ ] `ROLL` is set to your real roll number and every run used `set_seed()`
- [ ] `logs/` has one CSV per training run, including the two Part C runs
- [ ] `metrics.json` contains every number quoted in the report
- [ ] `figures/` has A1, A2, A3, B1, C1, C2, C3, D1, D2, E3
- [ ] the report states your held-out digit, your six digits and your attribute on line one
- [ ] the report has the **What surprised you** and **What did not work** sections
- [ ] you can open your Part C CSV at the viva and point at the row where the gradient died